# 18 — DeepSeek-V4 path (hash-MoE + C port)

**Before:** notebooks **11–17** (V2 + C backward).

**This notebook:** V4 building blocks — hash-MoE, SwiGLU, sliding attention map.

**Online course:** run cells **top-to-bottom**. Setup cell must print `data OK`.

**Dojo (optional):** `dojo-grade --lesson C2-L18`


**Before:** notebooks **11–17** (DeepSeek-V2 + C backward).

**This notebook:** V4 building blocks in PyTorch (`llmc/deepseek_v4.py`) and how they map to `c/`.

**Cursor workbook:** run cells top-to-bottom. Works from `llm-c-from-scratch/` or a student workspace with `pip install` curriculum. No full `make test_v4` inside the notebook (too slow) — use the fast pytest cell at the end.

Docs: `docs/V4_SOURCES_AND_SCOPE.md`, `docs/DEEPSEEK_VERSION_LADDER.md`.

**Dojo gate (optional):** after cells pass, run `dojo-grade --lesson C2-L18` as **Spm1CurorWorkbook**.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or pip-installed llmc) ---
import sys
from pathlib import Path

import torch

def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "deepseek_v4.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "deepseek_v4.py").is_file():
            return nested
    return Path.cwd()

ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data" / "tiny_shakespeare.txt"
C_DIR = ROOT / "c"
print("torch", torch.__version__)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing (optional for this notebook)")
print("c/", "OK" if (C_DIR / "Makefile").is_file() else "skip C smokes")


## V4 vs V2 (why a new track)

| Piece | DeepSeek-V2 (nb 12–17) | DeepSeek-V4 |
|-------|------------------------|-------------|
| Attention | MLA + dense causal | Sliding + CSA + HCA + indexer |
| FFN early layers | Routed MoE | **Hash-MoE** (token-id table) |
| FFN later layers | Routed MoE | Routed MoE (sqrt softplus gate) |
| Residual | `x + sublayer(x)` | **mHC** (hyper-connections) |

We port **one C file at a time** under `c/` — see phase map below.

In [ ]:
from llmc.deepseek_v4 import DeepSeekV4Config, build_hash_routing_table

# Tiny config — matches Dojo gate C2-L18 and pytest (CPU-friendly)
cfg = DeepSeekV4Config.tiny(vocab_size=64)
print("layer_types:", cfg.layer_types)
print("mlp_layer_types:", cfg.mlp_layer_types)

tid2eid = build_hash_routing_table(cfg)
print("tid2eid", tuple(tid2eid.shape), "token 7 -> experts", tid2eid[7].tolist())
print("(same formula as c/hash_moe.c)")


In [ ]:
from llmc.deepseek_v4 import SwiGLUExpert, HashMoE, RoutedMoE

b, t = 2, 8
ids = torch.randint(0, cfg.vocab_size, (b, t))
h = torch.randn(b, t, cfg.hidden_size)

# SwiGLU expert — matches c/swiglu.c (clamp + SiLU gate)
expert = SwiGLUExpert(cfg)
y_expert = expert(h.reshape(-1, cfg.hidden_size)).view(b, t, -1)
print("SwiGLU", tuple(h.shape), "->", tuple(y_expert.shape))

# Hash-MoE — bootstrap layers (token id -> expert table + gate weights)
hash_moe = HashMoE(cfg)
y_hash = hash_moe(h, ids)
print("HashMoE", tuple(h.shape), "->", tuple(y_hash.shape))

# Routed MoE — later V4 layers (learned top-k, no hash table)
routed = RoutedMoE(cfg)
y_route = routed(h)
print("RoutedMoE", tuple(h.shape), "->", tuple(y_route.shape))


## C port map (run in terminal, not in this notebook)

```text
c/rmsnorm.c, swiglu.c, hash_moe.c     phases 0–2
c/sliding_attn.c, v4_attention.c    phases 3–5
c/mhc.c, v4_layer.c, v4_model.c       phases 6–7
c/v4_train.c, train_deepseek_v4_tiny  phases 8+
```

From `llm-c-from-scratch/`:

```bash
cd c && make test_hash_moe    # fast — hash-MoE only
cd c && make test_v4          # full suite (minutes; CPU ok, no nvcc required for most targets)
pytest tests/test_deepseek_v4.py -q -k "hash_moe or swiglu"   # fast PyTorch + one C smoke
```

Optional nano reference: `./scripts/setup_vendor.sh` then re-run the optional cell below.

In [ ]:
# --- Fast verify (seconds) — same checks as Dojo gate C2-L18 ---
import subprocess

tests = ROOT / "tests" / "test_deepseek_v4.py"
if tests.is_file():
    cmd = [
        sys.executable,
        "-m",
        "pytest",
        str(tests),
        "-q",
        "-k",
        "hash_table or swiglu_forward or hash_moe_forward or routed_moe",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
    print("pytest OK — ready for dojo-grade --lesson C2-L18")
else:
    print("Skip pytest:", tests, "not found (pip install curriculum?)")


In [ ]:
# --- Optional: one fast C binary (skip if no gcc/make) ---
import shutil
import subprocess

RUN_C = False  # set True to build+run test_hash_moe (~30s first time)

if RUN_C and shutil.which("make") and (C_DIR / "Makefile").is_file():
    subprocess.run(["make", "bin/test_hash_moe"], cwd=str(C_DIR), check=True)
    out = subprocess.run(
        [str(C_DIR / "bin" / "test_hash_moe")], cwd=str(C_DIR), capture_output=True, text=True, check=True
    )
    print(out.stdout)
else:
    print("C smoke skipped (set RUN_C=True or run: cd c && make test_hash_moe)")


In [ ]:
# --- Optional: nano-deepseek-v4 full forward (large; needs vendor/) ---
VENDOR = ROOT / "vendor" / "nano-deepseek-v4"
if not VENDOR.is_dir():
    print("Skip nano: run ./scripts/setup_vendor.sh from", ROOT)
else:
    try:
        from llmc.deepseek_v4 import load_nano_model

        model = load_nano_model(cfg)
        model.eval()
        ids = torch.randint(0, cfg.vocab_size, (1, 8))
        with torch.no_grad():
            out = model(ids)
        logits = out.logits if hasattr(out, "logits") else out[0]
        print("nano forward logits", tuple(logits.shape))
    except Exception as e:
        print("nano optional failed:", e)


## Next steps

1. Terminal: `cd c && make test_v4` when you want the full C suite.
2. Dojo: `dojo-grade --discord-user-id YOUR_ID --lesson C2-L18` → paste `PASS-*` in Discord.
3. Read `c/v4_attention.c`, `sliding_attn.c` for attention phases; notebook 18 focused on **FFN bootstrap**.

**After this:** extend C train (`-train-e2e`), CUDA optional on Linux/GPU hosts.